# Neural Network Classifier

Ky notebook implementon pjesen e rrjetes neurale per klasifikimin e cmimit te shtepive ne tri klasa: `Low`, `Medium`, `High`.

Dataset-i origjinal ka target numerik `price`, prandaj origjinalisht mund te perdoret per regresion. Ne kete projekt problemi eshte kthyer ne klasifikim sepse target-i final eshte `price_class`, i krijuar ne notebook-un `04_scaling_train_val_test_split.ipynb`.

E rendesishme: ketu nuk krijojme target te ri dhe nuk perdorim `price` si target.

## 1. Importet

Per rrjeten neurale perdorim TensorFlow/Keras. Per vleresim perdorim metrikat standarde te klasifikimit nga scikit-learn.

In [1]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

sns.set_theme(style="whitegrid")

## 2. Konfigurimi dhe leximi i dataset-eve

Ky notebook pret qe dataset-et finale te jene krijuar paraprakisht nga notebook-u `04_scaling_train_val_test_split.ipynb`.

Nese mungojne `train_dataset.csv`, `val_dataset.csv` ose `test_dataset.csv`, duhet te ekzekutohet fillimisht notebook-u i preprocessing-ut. Nuk e ndryshojme logjiken e preprocessing-ut ketu.

In [2]:
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

BASE_DIR = Path.cwd().parent.parent if Path.cwd().name == "models" else Path.cwd()
DATA_DIR = BASE_DIR / "data" / "processed"

TRAIN_PATH = DATA_DIR / "train_dataset.csv"
VAL_PATH = DATA_DIR / "val_dataset.csv"
TEST_PATH = DATA_DIR / "test_dataset.csv"

for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Mungon file: {path}. "
            "Ekzekuto fillimisht notebook-un 04_scaling_train_val_test_split.ipynb "
            "qe te krijohen dataset-et finale."
        )

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df.columns = train_df.columns.str.strip()
val_df.columns = val_df.columns.str.strip()
test_df.columns = test_df.columns.str.strip()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (12810, 53)
Validation shape: (4267, 53)
Test shape: (4279, 53)


## 3. Pergatitja e features dhe target-it

Target-i eshte `price_class`. Kjo e ben problemin klasifikim, jo regresion.

Nese kolona `price` gjendet ne dataset, ajo largohet nga features per te shmangur data leakage, sepse `price_class` eshte krijuar nga `price`.

In [3]:
target_col = "price_class"

if target_col not in train_df.columns:
    raise ValueError(f"Mungon target-i i klasifikimit: {target_col}")

drop_cols = [target_col]

for col in ["price", "id", "date"]:
    if col in train_df.columns:
        drop_cols.append(col)

X_train = train_df.drop(columns=drop_cols)
X_val = val_df.drop(columns=drop_cols)
X_test = test_df.drop(columns=drop_cols)

y_train_raw = train_df[target_col]
y_val_raw = val_df[target_col]
y_test_raw = test_df[target_col]

class_mapping = {
    "Low": 0,
    "Medium": 1,
    "High": 2,
}

y_train = y_train_raw.map(class_mapping)
y_val = y_val_raw.map(class_mapping)
y_test = y_test_raw.map(class_mapping)

if y_train.isna().any() or y_val.isna().any() or y_test.isna().any():
    raise ValueError("Ka klasa te panjohura ne price_class. Priten vetem: Low, Medium, High.")

y_train = y_train.astype(int)
y_val = y_val.astype(int)
y_test = y_test.astype(int)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)
print("Classes:", class_mapping)
print("Columns removed from features:", drop_cols)

Train shape: (12810, 52)
Validation shape: (4267, 52)
Test shape: (4279, 52)
Classes: {'Low': 0, 'Medium': 1, 'High': 2}
Columns removed from features: ['price_class']


## 4. Shpjegim i arkitektures

Input layer merr te gjitha features numerike te dataset-it. Numri i neuroneve ne input percaktohet nga numri i kolonave ne `X_train`.

Hidden layers jane shtresat ku rrjeta meson lidhje me komplekse midis features dhe target-it. Aktivizimi `ReLU` perdoret sepse ndihmon rrjeten te mesoje lidhje jolineare dhe eshte i thjeshte per t'u trajnuar.

Output layer ka 3 neurone sepse kemi 3 klasa: `Low`, `Medium`, `High`. Aktivizimi `Softmax` perdoret sepse kthen probabilitet per secilen klase.

Loss function eshte `sparse_categorical_crossentropy`, sepse target-i eshte i koduar si numra te plote: `0`, `1`, `2`, jo si one-hot encoding.

## 5. Arkitekturat e rrjetes neurale

Testohen tri arkitektura:

- Architecture 1: rrjete me 2 hidden layers, `64` dhe `32` neurone.
- Architecture 2: rrjete me 3 hidden layers, `128`, `64`, `32` neurone, plus Dropout per te ulur overfitting.
- Architecture 3: rrjete me `BatchNormalization`, `Dropout` dhe `L2 regularization`, per te testuar nese stabilizimi dhe penalizimi i peshave permiresojne generalizimin.

In [4]:
def build_model_architecture_1(input_dim, learning_rate=0.001):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(64, activation="relu"),
        Dense(32, activation="relu"),
        Dense(3, activation="softmax"),
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model


def build_model_architecture_2(input_dim, learning_rate=0.001):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation="relu"),
        Dropout(0.3),
        Dense(64, activation="relu"),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(3, activation="softmax"),
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model


def build_model_architecture_3(input_dim, learning_rate=0.001):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation="relu", kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.25),
        Dense(64, activation="relu", kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.20),
        Dense(32, activation="relu"),
        Dropout(0.10),
        Dense(3, activation="softmax"),
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model

## 6. Funksioni per vleresim

Per cdo model llogariten metrikat kryesore te klasifikimit: accuracy, precision macro, recall macro dhe F1 macro.

In [5]:
def evaluate_model(model, X, y):
    y_pred_probs = model.predict(X, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)

    return {
        "accuracy": accuracy_score(y, y_pred),
        "precision_macro": precision_score(y, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y, y_pred, average="macro", zero_division=0),
    }

## 7. Eksperimentet me hiperparametra

Testohen tri learning rates dhe dy batch sizes per secilen arkitekture. Modeli trajnohet deri ne 100 epochs, por `EarlyStopping` e ndal trajnimin nese validation loss nuk permiresohet.

Arkitektura e trete shton edhe `BatchNormalization` dhe `L2 regularization`, prandaj krahasimi nuk mat vetem ndikimin e madhesise se rrjetes, por edhe ndikimin e teknikave per stabilizim dhe kontroll te overfitting.

In [ ]:
input_dim = X_train.shape[1]

experiments = [
    {
        "name": "Architecture 1 - 64/32",
        "builder": build_model_architecture_1,
    },
    {
        "name": "Architecture 2 - 128/64/32 + Dropout",
        "builder": build_model_architecture_2,
    },
    {
        "name": "Architecture 3 - 128/64/32 + BatchNorm + L2",
        "builder": build_model_architecture_3,
    },
]

learning_rates = [0.001, 0.0005, 0.0001]
batch_sizes = [32, 64]

results = []
histories = {}
trained_models = {}

for experiment in experiments:
    for learning_rate in learning_rates:
        for batch_size in batch_sizes:
            print("=" * 70)
            print(f"Training: {experiment['name']}")
            print(f"Learning rate: {learning_rate}, Batch size: {batch_size}")

            tf.keras.backend.clear_session()
            tf.random.set_seed(RANDOM_STATE)

            model = experiment["builder"](input_dim, learning_rate)

            early_stopping = EarlyStopping(
                monitor="val_loss",
                patience=10,
                restore_best_weights=True,
            )

            history = model.fit(
                X_train,
                y_train,
                validation_data=(X_val, y_val),
                epochs=100,
                batch_size=batch_size,
                callbacks=[early_stopping],
                verbose=1,
            )

            val_metrics = evaluate_model(model, X_val, y_val)

            result = {
                "architecture": experiment["name"],
                "learning_rate": learning_rate,
                "batch_size": batch_size,
                "epochs_trained": len(history.history["loss"]),
                "val_accuracy": val_metrics["accuracy"],
                "val_precision_macro": val_metrics["precision_macro"],
                "val_recall_macro": val_metrics["recall_macro"],
                "val_f1_macro": val_metrics["f1_macro"],
            }

            results.append(result)

            model_key = f"{experiment['name']}_lr{learning_rate}_bs{batch_size}"
            histories[model_key] = history
            trained_models[model_key] = model

results_df = pd.DataFrame(results)
results_df.sort_values(by="val_f1_macro", ascending=False)

## 8. Zgjedhja e modelit me te mire

Modeli me i mire zgjidhet sipas `validation macro F1`, sepse kjo metrike i trajton te tri klasat me rendesi te barabarte.

In [ ]:
best_result = results_df.sort_values(by="val_f1_macro", ascending=False).iloc[0]

best_model_key = (
    f"{best_result['architecture']}_"
    f"lr{best_result['learning_rate']}_"
    f"bs{best_result['batch_size']}"
)

best_model = trained_models[best_model_key]
best_history = histories[best_model_key]
best_model_title = (
    f"{best_result['architecture']} | "
    f"lr={best_result['learning_rate']} | "
    f"batch={best_result['batch_size']}"
)

print("Best Neural Network model:")
print(best_result)

## 9. Grafiket e trajnimit

Krahasimi i training loss me validation loss dhe training accuracy me validation accuracy ndihmon te shohim nese modeli ka overfitting. Nese training performanca vazhdon te permiresohet ndersa validation performanca perkeqesohet, atehere modeli po ben overfitting.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(best_history.history["loss"], label="Training Loss")
plt.plot(best_history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title(f"Training Loss vs Validation Loss\n{best_model_title}")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(best_history.history["accuracy"], label="Training Accuracy")
plt.plot(best_history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title(f"Training Accuracy vs Validation Accuracy\n{best_model_title}")
plt.legend()
plt.grid(True)
plt.show()

## 10. Vleresimi final ne test set

Pasi modeli me i mire u zgjodh nga validation set, vleresimi final behet vetem nje here ne test set.

In [ ]:
y_test_pred_probs = best_model.predict(X_test, verbose=0)
y_test_pred = np.argmax(y_test_pred_probs, axis=1)

test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, average="macro", zero_division=0)
test_recall = recall_score(y_test, y_test_pred, average="macro", zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, average="macro", zero_division=0)

print("Neural Network - Test Results")
print("Accuracy:", test_accuracy)
print("Precision macro:", test_precision)
print("Recall macro:", test_recall)
print("F1 macro:", test_f1)

In [ ]:
class_names = ["Low", "Medium", "High"]

print(classification_report(
    y_test,
    y_test_pred,
    target_names=class_names,
    zero_division=0,
))

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title(f"Confusion Matrix\n{best_model_title}")
plt.show()

## 11. Tabela krahasuese per Neural Network

Tabela e pare tregon te gjitha eksperimentet ne validation set, te renditura sipas `validation F1 macro`. Grafiku krahasues tregon vizualisht cilet konfigurime performuan me mire. Tabela e dyte tregon rezultatin final te modelit me te mire ne test set.

In [ ]:
nn_experiment_results = results_df.sort_values(by="val_f1_macro", ascending=False).reset_index(drop=True)
display(nn_experiment_results)

plot_df = nn_experiment_results.copy()
plot_df["configuration"] = (
    plot_df["architecture"]
    + " | lr=" + plot_df["learning_rate"].astype(str)
    + " | batch=" + plot_df["batch_size"].astype(str)
)

plt.figure(figsize=(12, 8))
sns.barplot(
    data=plot_df,
    x="val_f1_macro",
    y="configuration",
    hue="architecture",
    dodge=False,
)
plt.xlabel("Validation F1 Macro")
plt.ylabel("Configuration")
plt.title("Neural Network Hyperparameter Comparison")
plt.legend(title="Architecture", loc="lower right")
plt.grid(True, axis="x")
plt.tight_layout()
plt.show()

In [ ]:
nn_final_results = pd.DataFrame([
    {
        "model": "Neural Network",
        "best_architecture": best_result["architecture"],
        "learning_rate": best_result["learning_rate"],
        "batch_size": best_result["batch_size"],
        "epochs_trained": best_result["epochs_trained"],
        "val_accuracy": best_result["val_accuracy"],
        "val_precision_macro": best_result["val_precision_macro"],
        "val_recall_macro": best_result["val_recall_macro"],
        "val_f1_macro": best_result["val_f1_macro"],
        "test_accuracy": test_accuracy,
        "test_precision_macro": test_precision,
        "test_recall_macro": test_recall,
        "test_f1_macro": test_f1,
    }
])

nn_final_results

## 12. Perfundim

Neural Network u perdor si klasifikues per `price_class`, pra per te parashikuar nese nje shtepi i takon klases `Low`, `Medium` ose `High`. Modeli nuk e perdor kolonen `price` si input, sepse `price_class` eshte krijuar nga `price` dhe perfshirja e saj do te shkaktonte data leakage. Si input perdoren karakteristikat origjinale te shtepive dhe features te krijuara gjate feature engineering.

U testuan tri arkitektura kryesore. Arkitektura e pare ishte me e thjeshte, me dy hidden layers (`64/32`). Arkitektura e dyte ishte me e thelle, me tri hidden layers (`128/64/32`) dhe perdori `Dropout` per te ulur rrezikun e overfitting. Arkitektura e trete perdori te njejten ide baze `128/64/32`, por shtoi `BatchNormalization` dhe `L2 regularization` per te testuar nese stabilizimi i trajnimit dhe penalizimi i peshave permiresojne generalizimin. Ne hidden layers u perdor aktivizimi `ReLU`, sepse ndihmon modelin te mesoje lidhje jolineare mes features dhe target-it. Ne output layer u perdor `Softmax`, sepse problemi ka tri klasa dhe modeli duhet te jape probabilitet per secilen klase.

Rezultatet ne validation set treguan se modeli me i mire ishte `Architecture 2 - 128/64/32 + Dropout`, me `learning_rate = 0.0001` dhe `batch_size = 32`. Ky konfigurim u trajnua per `100` epochs dhe arriti `validation accuracy = 0.842512`, `validation precision macro = 0.843581`, `validation recall macro = 0.841861` dhe `validation F1 macro = 0.842515`. Modeli me i mire u zgjodh duke renditur rezultatet sipas `validation F1 macro`, jo sipas kompleksitetit te arkitektures.

Krahasimi mes arkitekturave tregon se arkitektura e dyte ishte me e pershtatshme per kete dataset. Ajo performoi me mire se arkitektura e pare me `64/32`, qe sherbeu si baseline, dhe gjithashtu me mire se arkitektura e trete me `BatchNormalization` dhe `L2 regularization`. Kjo sugjeron se ne kete rast `Dropout` ishte i mjaftueshem per kontrollimin e overfitting, ndersa shtimi i `BatchNormalization` dhe `L2` nuk solli permiresim shtese ne validation set.

Nga hiperparametrat, `learning_rate = 0.0001` me `batch_size = 32` dha rezultatin me te mire. Edhe pse ky konfigurim kerkoi me shume epochs dhe arriti deri ne kufirin maksimal prej `100` epochs, ai dha performancen me te larte ne validation set. Kjo tregon se perditesimet me te vogla te peshave mund ta kene ndihmuar modelin te mesoje me gradualisht dhe te arrije generalizim pak me te mire.

Arkitektura e pare (`64/32`) pati rezultate me te uleta, me `validation F1 macro` rreth `0.828 - 0.834`. Arkitektura e trete arriti rezultatin me te mire te saj me `validation F1 macro = 0.839000`, por prape mbeti poshte arkitektures se dyte. Prandaj, kompleksiteti shtese nuk ishte automatikisht me i mire; arkitektura me e suksesshme ishte ajo qe kishte balancen me te mire mes kapacitetit dhe regularizimit.

Modeli me i mire u zgjodh sipas `validation macro F1`, sepse kjo metrike i trajton klasat `Low`, `Medium` dhe `High` me rendesi te barabarte. Kjo eshte me e pershtatshme se vetem accuracy, sidomos nese klasat nuk jane plotesisht te balancuara. Modeli final u vleresua me accuracy, precision, recall, F1-score, classification report dhe confusion matrix.

Ne pergjithesi, rezultatet tregojne se Neural Network eshte i pershtatshem per kete problem klasifikimi dhe se arkitektura `128/64/32 + Dropout` ishte zgjedhja me e mire nga konfigurimet e testuara. Rezultati me i mire ne validation set ishte `F1 macro = 0.842515`, qe tregon performance te mire dhe relativisht te balancuar ne klasat `Low`, `Medium` dhe `High`.